In [ ]:
import pandas as pd
import numpy as np
import os, sys

class Classify():

    print_columns_d = ['index', 'Object', 'Sanitised', 'Reduced', 'RA', 'DEC', 
                    'SNR', 'MJD-OBS', 'EXPTIME', 'New Groups', 'Min_SNR', 'RV_pos' ]

    print_columns_o = ['index', 'Object', 'Sanitised', 'Reduced', 'RA', 'DEC', 'SNR', 'MJD-OBS',
     'EXPTIME', 'New Groups', 'Min_SNR', 'RV_pos', 'Abs_width', 'Abs_depth' ]

    def __init__(self, parameters, res_path):
        
        self.param = [parameters["threshold"], parameters["med_filter_bin"],
                        parameters["rv_min"], parameters["rv_max"], parameters["cutoff"],
                         parameters["width_filt"]]

        self.res_path = res_path
        self.cand_report_path = res_path + 'candidate_report.pkl'

        status = ['flagged', 'not_candidate_but_real', 'not_candidate_but_junk', 'candidate', 'not_candidate']

        for st in status:
            if os.path.exists(res_path + '{}/'.format(st)) == False:
                os.mkdir(res_path + '{}/'.format(st))

        if os.path.exists(self.cand_report_path):
            self.candidate_report = pd.read_pickle(self.cand_report_path)
        else:
            self.candidate_report = pd.DataFrame(data={'Target':[], 'Status':[], 'Parameters':[]}).astype(object)

        self.target_name = None
        self.target_info = None
        self.detection_info = None
        self.previous_report = None
        self.current_status = None
        self.current_destination = None

        self.skipped = []
        self.flagged = False

    def _normalise_destination(self, destination):
        """Normalise a user-defined destination under the result directory."""
        cleaned = destination.strip().replace('\\', '/').strip('/')
        parts = [p for p in cleaned.split('/') if p not in ('', '.')]
        if len(parts) == 0:
            return None
        if '..' in parts:
            raise ValueError('Parent directory traversal is not allowed.')
        return '/'.join(parts)

    def _set_route(self, status, destination=None):
        self.current_status = status
        self.current_destination = self._normalise_destination(destination) if destination is not None else None

    def _best_row(self, frame):
        if frame is None or len(frame) == 0 or 'Min_SNR' not in frame.columns:
            return None

        min_snr = pd.to_numeric(frame['Min_SNR'], errors='coerce')
        if min_snr.isna().all():
            return None

        return frame.loc[min_snr.idxmin()]

    def auto_route(self, has_detection, rv_min=None, rv_max=None, threshold=None, width_filter=None):
        def to_float(value):
            converted = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
            if pd.isna(converted):
                return np.nan
            return float(converted)

        def at_edge(rv_value):
            if np.isnan(rv_value) or rv_min is None or rv_max is None:
                return False

            edge_margin = max(25.0, 0.1 * abs(rv_max - rv_min))
            return (rv_value <= rv_min + edge_margin) or (rv_value >= rv_max - edge_margin)

        source = self.detection_info if has_detection else self.target_info
        best_row = self._best_row(source)

        if best_row is None:
            self._set_route('not_candidate_but_junk', 'not_candidate/no_detection')
            print('{}: AUTO -> not_candidate/no_detection'.format(self.target_name))
            return

        min_snr = to_float(best_row.get('Min_SNR'))
        rv_pos = to_float(best_row.get('RV_pos'))
        width = to_float(best_row.get('Abs_width'))
        depth = to_float(best_row.get('Abs_depth'))
        near_edge = at_edge(rv_pos)

        if not has_detection or self.detection_info is None or len(self.detection_info) == 0:
            if near_edge and np.isfinite(min_snr) and threshold is not None and min_snr <= threshold + 1.0:
                self._set_route('not_candidate_but_junk', 'not_candidate/edge_noise')
            elif np.isfinite(min_snr) and threshold is not None and min_snr <= threshold + 1.0:
                self._set_route('not_candidate_but_junk', 'not_candidate/subthreshold_absorption')
            else:
                self._set_route('not_candidate_but_junk', 'not_candidate/no_detection')

            print('{}: AUTO -> {}'.format(self.target_name, self.current_destination))
            return

        detection_count = len(self.detection_info)
        strong_signal = threshold is not None and np.isfinite(min_snr) and min_snr <= threshold - 2.0
        repeated_signal = detection_count >= 2
        broad_signal = (np.isfinite(width) and width_filter is not None and width >= width_filter + 5) or (np.isfinite(depth) and depth >= 0.2)
        near_centre = np.isfinite(rv_pos) and abs(rv_pos) <= 25.0

        if near_edge:
            self._set_route('flagged', 'candidate/review_edge_of_window')
        elif repeated_signal and near_centre and broad_signal and not strong_signal:
            self._set_route('not_candidate_but_real', 'not_candidate/complex_variability')
        elif repeated_signal and strong_signal:
            self._set_route('candidate', 'candidate/repeated_strong')
        elif strong_signal:
            self._set_route('candidate', 'candidate/single_epoch_strong')
        elif repeated_signal:
            self._set_route('flagged', 'candidate/review_repeated_marginal')
        else:
            self._set_route('flagged', 'candidate/review_single_epoch_marginal')

        print('{}: AUTO -> {}'.format(self.target_name, self.current_destination))

    def candidate_info(self, target):
        ''' Load all information from a specific target.
            Return -> Has the target already been looked at? True/False '''
        
        self.current_status = None
        self.current_destination = None
        self.target_name = target

        self.previous_report = self.candidate_report[self.candidate_report['Target'] == target]

        if len(self.previous_report) == 1:
            classified = True
            if 'flagged' in self.previous_report.Status.to_numpy():
                self.flagged = True
            else:
                self.flagged = False
        elif len(self.previous_report) > 1:
            raise ValueError('Candidate:{} has more than one entry in the DataFrame.'.format(target))
        else:
            classified = False
            self.flagged = False

        return classified

    def classify(self):
        ''' Classify target according to its status - ie. candidate/not candidate/flagged'''

        if self.current_status == 'skipped':
            self.skipped.append(self.target_name)
            return None

        elif self.flagged is not True:
            cand_report = self.candidate_report
            cand_report.loc[len(cand_report.index)] = [self.target_name, self.current_status, self.param]
            self.candidate_report = cand_report.astype(object)

        else:
            self.candidate_report.iloc[self.previous_report.index.values[0]]['Status'] = self.current_status
        
        save_dir = os.path.join(self.res_path, self.current_destination or self.current_status)
        os.makedirs(save_dir, exist_ok=True)
        return save_dir + '/'

    def ask_user(self):
        ''' Prompt the user in the notebook cell output for a classification.
            Keys: y=candidate, n=not candidate (real), j=junk, w=flag, space/s=skip,
                  enter/q=quick save, esc/x=quit+save, f=custom folder, d=print df, o=print detections '''
        print('\nClassify {} — y=candidate | n=not cand (real) | j=junk | w=flag | s=skip | f=custom folder | q=save | x=quit+save | d=print df | o=print detections'.format(self.target_name))
        while True:
            key = input('>>> ').strip().lower()

            if key == 'y':
                self._set_route('candidate')
                print('{}: CANDIDATE'.format(self.target_name))
                break

            elif key == 'n':
                self._set_route('not_candidate_but_real')
                print('{}: NOT A CANDIDATE — real astrophysical variability'.format(self.target_name))
                break

            elif key == 'j':
                self._set_route('not_candidate_but_junk')
                print('{}: NOT A CANDIDATE — junk'.format(self.target_name))
                break

            elif key == 'w':
                self._set_route('flagged')
                print('{}: FLAGGED'.format(self.target_name))
                break

            elif key in ('s', ''):
                self._set_route('skipped')
                print('{}: SKIPPED'.format(self.target_name))
                break

            elif key == 'f':
                while True:
                    destination = input('Destination folder under results (e.g. FalsePositive/RadialVelocityShift): ').strip()
                    try:
                        destination = self._normalise_destination(destination)
                    except ValueError as err:
                        print(err)
                        continue

                    if destination is None:
                        print('Destination cannot be empty. Try again.')
                        continue

                    self._set_route(destination, destination)
                    print('{}: SENT TO {}'.format(self.target_name, destination))
                    break
                break

            elif key == 'q':
                print('Saving progress...')
                self.candidate_report.to_pickle(self.cand_report_path)
                self.candidate_report.to_html(self.res_path + 'Report.html')
                print('Saved!')

            elif key == 'x':
                self.candidate_report.to_pickle(self.cand_report_path)
                self.candidate_report.to_html(self.res_path + 'Report.html')
                print('Saved! Quitting.')
                sys.exit()

            elif key == 'd':
                avail = [c for c in self.print_columns_d if c in self.target_info.columns]
                print(self.target_info[avail])

            elif key == 'o':
                if self.detection_info is not None:
                    avail = [c for c in self.print_columns_o if c in self.detection_info.columns]
                    print(self.detection_info[avail])
                else:
                    print('No detection info available.')

            else:
                print('Unknown key "{}". Try again.'.format(key))

    def onkey(self, event):
        ''' Legacy matplotlib key-press handler (not used in notebook mode). '''
        key_map = {
            'y': ('candidate', '{}: Currently CANDIDATE'),
            'n': ('not_candidate_but_real', '{}: Currently NOT A CANDIDATE: Real astrophysical variability'),
            'j': ('not_candidate_but_junk', '{}: Currently NOT A CANDIDATE: Junk'),
            'w': ('flagged', '{}: Currently FLAGGED'),
            ' ': ('skipped', '{}: Currently SKIPPED'),
        }
        if event.key in key_map:
            status, msg = key_map[event.key]
            self._set_route(status)
            print(msg.format(self.target_name))
        elif event.key == 'enter':
            print('Saving progress...')
            self.candidate_report.to_pickle(self.cand_report_path)
            self.candidate_report.to_html(self.res_path + 'Report.html')
            print('Saved!')
        elif event.key == 'escape':
            self.candidate_report.to_pickle(self.cand_report_path)
            self.candidate_report.to_html(self.res_path + 'Report.html')
            print('Saved!')
            sys.exit()
        elif event.key == 'd':
            avail = [c for c in self.print_columns_d if c in self.target_info.columns]
            print(self.target_info[avail])
        elif event.key == 'o':
            if self.detection_info is not None:
                avail = [c for c in self.print_columns_o if c in self.detection_info.columns]
                print(self.detection_info[avail])
            else:
                print('No detection info available.')